# 02 — Dispatching Simulation

运行离散事件仿真，比较 FIFO、SPT、EDD、CR 四种派工规则。

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from simulator import run_simulation, run_all_rules, _load_operations, _load_lot_data
from metrics import print_comparison, bottleneck_ranking_from_events
from visualization import (
    plot_gantt, plot_rule_comparison, plot_utilization_heatmap,
    plot_bottleneck_ranking, plot_queue_distribution
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [2]:
# Load data
ops = _load_operations('../data/processed/operations.csv')
lots = _load_lot_data('../data/processed/dim_lot.csv')
print(f'Loaded {len(ops)} operations, {len(lots)} lots')

Loaded 1100 operations, 80 lots


## Run All 4 Rules (Baseline Scenario)

In [3]:
all_events, summaries = run_all_rules(ops, lots, ['FIFO', 'SPT', 'EDD', 'CR'], scenario_id='baseline')

  [FIFO  ] makespan=   82.25h  avg_cycle=  54.89h  on_time=100.0%  bottleneck=T01
  [SPT   ] makespan=  118.02h  avg_cycle=  60.54h  on_time=100.0%  bottleneck=T01
  [EDD   ] makespan=  119.52h  avg_cycle=  66.78h  on_time=100.0%  bottleneck=T01
  [CR    ] makespan=   83.87h  avg_cycle=  62.98h  on_time=100.0%  bottleneck=T01


In [4]:
# Comparison table
print_comparison(summaries)

--------------------------------------------------------------------------------------
Rule       Makespan   AvgCycle   AvgQueue     Q%  OnTime%   Util%    BN_Tool  BN_Util%
--------------------------------------------------------------------------------------
FIFO         82.2h     54.9h     47.4h  85.9%   100.0%   48.8%        T01     60.5%
CR           83.9h     63.0h     55.5h  87.8%   100.0%   47.8%        T01     59.4%
SPT         118.0h     60.5h     53.0h  86.6%   100.0%   34.0%        T01     42.2%
EDD         119.5h     66.8h     59.3h  87.5%   100.0%   33.6%        T01     41.7%
--------------------------------------------------------------------------------------


In [5]:
# Detailed summary DataFrame
df_summary = pd.DataFrame(summaries)
df_summary[[
    'rule_name', 'makespan', 'avg_cycle_time', 'avg_queue_time',
    'queue_time_ratio', 'on_time_rate', 'avg_tool_utilization',
    'bottleneck_tool', 'bottleneck_utilization'
]]

,rule_name,makespan,avg_cycle_time,avg_queue_time,queue_time_ratio,on_time_rate,avg_tool_utilization,bottleneck_tool,bottleneck_utilization
0,FIFO,82.25,54.89,47.36,85.9,100.0,48.8,T01,60.5
1,SPT,118.02,60.54,53.02,86.6,100.0,34.0,T01,42.2
2,EDD,119.52,66.78,59.26,87.5,100.0,33.6,T01,41.7
3,CR,83.87,62.98,55.46,87.8,100.0,47.8,T01,59.4


## Gantt Chart — FIFO Baseline

In [6]:
plot_gantt(all_events['FIFO'], rule_name='FIFO', max_lots=10, save=False)
plt.show()

## Rule Comparison Charts

In [7]:
plot_rule_comparison(summaries, save=False)
plt.show()

## Tool Utilization Heatmap — FIFO

In [8]:
plot_utilization_heatmap(all_events['FIFO'], rule_name='FIFO', save=False)
plt.show()

## Bottleneck Ranking

In [9]:
ranking = bottleneck_ranking_from_events(all_events['FIFO'])
pd.DataFrame(ranking).head(10)

,entity,utilization_pct,avg_queue_hours,total_wip,bottleneck_score,rank
0,T07,59.0,6.92,80,0.9814,1
1,T02,60.3,4.82,80,0.8610,2
2,T09,56.3,4.65,80,0.7994,3
3,T04,56.3,4.63,80,0.7991,4
4,T01,60.5,2.31,80,0.7004,5
5,T00,56.8,2.85,80,0.6887,6
6,T05,59.7,1.84,80,0.6590,7
7,T08,58.9,1.55,80,0.6305,8
8,T06,57.2,1.53,80,0.6079,9
9,T03,58.0,0.77,80,0.5688,10


In [10]:
plot_bottleneck_ranking(ranking, title_suffix=' — FIFO', save=False)
plt.show()

## Queue Time Distribution

In [11]:
plot_queue_distribution(all_events['FIFO'], rule_name='FIFO', save=False)
plt.show()

## Business Questions Answered

1. **Which rule gives the lowest makespan?** → Compare the makespan column.
2. **Which rule gives the lowest average cycle time?** → Check avg_cycle_time.
3. **Which rule gives the best on-time rate?** → Check on_time_rate.
4. **Which tool is the bottleneck?** → Check bottleneck_tool and ranking chart.
5. **If the business objective differs, which rule to choose?**
   - Minimise cycle time → SPT
   - Maximise on-time delivery → EDD or CR
   - Balance and simplicity → FIFO